In [1]:
from dataclasses import dataclass

In [2]:
import numpy as np

In [3]:
from resprops import Fluid, Layer
from prodpy.porsim.prerun import Borehole
from prodpy.transient._base_solver import BaseSolver

In [4]:
@dataclass
class Table3Parameters:
    q_w: float = 184.44          # flux at surface, bbl/day
    dp_wf: float = 500           # pressure drop at the wellbore, psi
    phi: float = 0.25            # porosity, fraction
    c_t: float = 4.20e-6         # total compressibility, psi^-1
    h: float = 107               # payzone thickness, ft
    r_w: float = 0.29            # wellbore radius, ft
    r_e: float = 2900            # reservoir radius, ft
    mu: float = 2.5              # viscosity, cp
    k: float = 10.9              # permeability, mD
    V_w: float = 2210            # wellbore volume, bbl
    S: float = 7.7               # skin factor
    x_f: float = 250             # fracture half-length, ft

In [5]:
fluid = Fluid(
    visc = Table3Parameters.mu
); fluid

Fluid(visc=2.5, rho=62.4, comp=1e-06, fvf=1.0, press=None, satur=1.0, rperm=1.0)

In [6]:
layer = Layer(
    Table3Parameters.k,
    poro = Table3Parameters.phi,
); layer

Layer(perm=10.9, is_isotropic=True, poro=0.25, comp=None, press=None)

In [7]:
well = Borehole(
    radius = Table3Parameters.r_w,
    skin = Table3Parameters.S,
)

well.add_constraint(orate=Table3Parameters.q_w); well

Borehole(index=None, radius=0.29 ft, skin=7.7, constraints=1)

In [8]:
for index, c in enumerate(well.constraints):
    print(f"Constraint {index}: {c}")

Constraint 0: Constraint(start=0, stop=None, orate=184.44)


In [9]:
solver = BaseSolver(
    np.pi * Table3Parameters.r_e**2 / BaseSolver.ACRE_TO_SQFT,
    Table3Parameters.h,
    fluid = fluid,
    layer = layer,
    borehole = well,
    tcomp = Table3Parameters.c_t,
); solver

BaseSolver(reservoir_area=606.5 acres, payzone_height=107.0 ft, shape_key='circle_center')

In [13]:
solver._hdiff

array([0.02825528])

In [ ]:
class ExponentialIntegralSolution:
    def __init__(self, params: Table3Parameters):
        self.params = params

    def beta(self, radius:np.ndarray, time:float):
        return radius**2 / (4 * self.hydraulic_diffusivity * time)